# ACE26 Chat Playground Backup Notebook

This notebook is a lightweight backup version of the ACE26 workshop playground. It mirrors the activity guide at a code level: edit instructions, adjust reasoning effort and max completion tokens, optionally retrieve workshop document snippets, and call the configured Azure OpenAI deployment.

Use `USE_LIVE_AZURE = False` for a no-service walkthrough with canned sample snippets. Use `USE_LIVE_AZURE = True` when the workshop Azure AI Services and Azure AI Search resources are available.

## 1. Imports

The notebook uses Python standard-library HTTP calls so it can run in a plain Jupyter or Colab runtime without installing Azure SDK packages.

In [ ]:
import json
import os
import subprocess
import textwrap
import urllib.error
import urllib.request
from getpass import getpass

def pretty_print(text, width=100):
    print(textwrap.fill(str(text), width=width, replace_whitespace=False))

def trim(value, max_chars=1400):
    value = " ".join(str(value or "").split())
    return value if len(value) <= max_chars else value[: max_chars - 3] + "..."

## 2. Configuration

For live Azure mode, fill in the endpoints and deployment names. Keep keys private. In Colab, use the prompt cell below rather than hard-coding keys.

In [ ]:
USE_LIVE_AZURE = False

AZURE_AI_SERVICES_ENDPOINT = os.getenv("AZURE_AI_SERVICES_ENDPOINT", "")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", os.getenv("CHAT_DEPLOYMENT_NAME", "gpt-5.4-pro"))
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "v1")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_AUTH_TOKEN = os.getenv("AZURE_OPENAI_AUTH_TOKEN", "")

AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "")
AZURE_SEARCH_INDEX = os.getenv("AZURE_SEARCH_INDEX", "documents")
AZURE_SEARCH_API_VERSION = os.getenv("AZURE_SEARCH_API_VERSION", "2024-07-01")
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY", "")
AZURE_SEARCH_AUTH_TOKEN = os.getenv("AZURE_SEARCH_AUTH_TOKEN", "")

DEFAULT_INSTRUCTIONS = """You are an AI assistant for the ACE26 water and environmental workshop.
Help participants learn prompt iteration, model settings, and document-grounded answers.
Use clear language, be explicit about uncertainty, and when retrieved source snippets are supplied cite them as [source 1], [source 2], etc."""

print("Live Azure mode:", USE_LIVE_AZURE)
print("Chat deployment:", AZURE_OPENAI_CHAT_DEPLOYMENT)
print("Search index:", AZURE_SEARCH_INDEX)

In [ ]:
# Optional secure prompts for live mode. Run this cell only when USE_LIVE_AZURE is True.
if USE_LIVE_AZURE:
    AZURE_AI_SERVICES_ENDPOINT = AZURE_AI_SERVICES_ENDPOINT or input("Azure AI Services endpoint: ").strip().rstrip("/")
    AZURE_OPENAI_CHAT_DEPLOYMENT = AZURE_OPENAI_CHAT_DEPLOYMENT or input("Chat deployment name: ").strip()
    AZURE_SEARCH_ENDPOINT = AZURE_SEARCH_ENDPOINT or input("Azure AI Search endpoint: ").strip().rstrip("/")
    AZURE_SEARCH_INDEX = AZURE_SEARCH_INDEX or input("Azure AI Search index name [documents]: ").strip() or "documents"

    if not AZURE_OPENAI_API_KEY and not AZURE_OPENAI_AUTH_TOKEN:
        AZURE_OPENAI_API_KEY = getpass("Azure OpenAI/API key, or leave blank to try Azure CLI token: ")
    if not AZURE_SEARCH_API_KEY and not AZURE_SEARCH_AUTH_TOKEN:
        AZURE_SEARCH_API_KEY = getpass("Azure AI Search query/admin key, or leave blank to try Azure CLI token: ")

print("Configuration cell complete.")

## 3. Authentication Helpers

Live mode supports API keys, explicit bearer tokens, or Azure CLI tokens from a local Jupyter environment. Colab users usually use API keys for this backup notebook.

In [ ]:
def azure_cli_token(resource):
    output = subprocess.check_output([
        "az", "account", "get-access-token", "--resource", resource, "--output", "json"
    ], text=True)
    return json.loads(output)["accessToken"]

def openai_headers():
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    if AZURE_OPENAI_API_KEY:
        headers["api-key"] = AZURE_OPENAI_API_KEY
    else:
        token = AZURE_OPENAI_AUTH_TOKEN or azure_cli_token("https://ai.azure.com")
        headers["Authorization"] = f"Bearer {token}"
    return headers

def search_headers():
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    if AZURE_SEARCH_API_KEY:
        headers["api-key"] = AZURE_SEARCH_API_KEY
    else:
        token = AZURE_SEARCH_AUTH_TOKEN or azure_cli_token("https://search.azure.com")
        headers["Authorization"] = f"Bearer {token}"
    return headers

def request_json(method, url, headers, body=None):
    data = None if body is None else json.dumps(body).encode("utf-8")
    request = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            text = response.read().decode("utf-8")
            return json.loads(text) if text else {}
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"{method} {url} failed with {exc.code}: {detail}") from exc

## 4. Search and Chat Helpers

These functions are intentionally similar to the web app API: retrieve snippets, add them to the instructions, then call the chat completion endpoint.

In [ ]:
OFFLINE_SOURCES = [
    {
        "title": "Fort Worth FY2021-2025 Adopted CIP",
        "sourceFile": "fort-worth-fy2021-2025-adopted-cip.pdf",
        "page": None,
        "chunk": 1,
        "content": "Major Mains Bucket includes installation, repair, and replacement of large transmission mains."
    },
    {
        "title": "Fort Worth FY2021-2025 Adopted CIP",
        "sourceFile": "fort-worth-fy2021-2025-adopted-cip.pdf",
        "page": None,
        "chunk": 2,
        "content": "WTP Minor Improvements includes upgrades at Eagle Mountain, North Holly, South Holly, Rolling Hills, and Westside water treatment plants."
    },
    {
        "title": "Fort Worth FY2021-2025 Adopted CIP",
        "sourceFile": "fort-worth-fy2021-2025-adopted-cip.pdf",
        "page": None,
        "chunk": 3,
        "content": "Miscellaneous Water Facilities includes storage tanks, pump stations, and wholesale customer water meter stations."
    },
    {
        "title": "Fort Worth FY2021-2025 Adopted CIP",
        "sourceFile": "fort-worth-fy2021-2025-adopted-cip.pdf",
        "page": None,
        "chunk": 4,
        "content": "Water Sewer Program - Streets includes replacement of aging water and sewer lines during street reconstruction."
    }
]

def search_documents(query, top=4):
    if not USE_LIVE_AZURE:
        return OFFLINE_SOURCES[:top]
    if not AZURE_SEARCH_ENDPOINT or not AZURE_SEARCH_INDEX:
        return []
    url = f"{AZURE_SEARCH_ENDPOINT.rstrip('/')}/indexes/{AZURE_SEARCH_INDEX}/docs/search?api-version={AZURE_SEARCH_API_VERSION}"
    body = {
        "search": query or "*",
        "queryType": "simple",
        "searchMode": "any",
        "top": top,
        "select": "title,sourceFile,page,chunk,content"
    }
    response = request_json("POST", url, search_headers(), body)
    return [
        {
            "title": item.get("title") or item.get("sourceFile") or "Workshop document",
            "sourceFile": item.get("sourceFile", ""),
            "page": item.get("page"),
            "chunk": item.get("chunk"),
            "content": trim(item.get("content", ""))
        }
        for item in response.get("value", [])
    ]

def build_system_prompt(instructions, sources):
    if not sources:
        return instructions
    source_text = []
    for index, source in enumerate(sources, start=1):
        location = f" page {source['page']}" if source.get("page") else ""
        source_text.append(f"[source {index}] {source.get('title', 'Workshop document')}{location}\n{source.get('content', '')}")
    return instructions + "\n\nUse these retrieved workshop snippets when relevant. If they do not support the answer, say what is missing instead of guessing.\n\n" + "\n\n".join(source_text)

def offline_answer(prompt, sources):
    if "table" in prompt.lower():
        return "| category | description | source |\n| --- | --- | --- |\n| WTP Minor Improvements | Treatment plant upgrades | [source 2] |\n| Major Mains Bucket | Large transmission main work | [source 1] |"
    if "wtp minor" in prompt.lower():
        return "WTP Minor Improvements refers to upgrades at Fort Worth water treatment plants including Eagle Mountain, North Holly, South Holly, Rolling Hills, and Westside. [source 2]"
    return "Offline sample: retrieved snippets point to major mains, water treatment plant improvements, water facilities, and street-related water/sewer replacement work. Use live Azure mode for model-generated wording and citations."

def chat(prompt, instructions=DEFAULT_INSTRUCTIONS, use_grounding=True, reasoning_effort="medium", max_completion_tokens=900, top=4):
    sources = search_documents(prompt, top=top) if use_grounding else []
    if not USE_LIVE_AZURE:
        return {"answer": offline_answer(prompt, sources), "sources": sources, "usage": None}

    if not AZURE_AI_SERVICES_ENDPOINT or not AZURE_OPENAI_CHAT_DEPLOYMENT:
        raise RuntimeError("Set AZURE_AI_SERVICES_ENDPOINT and AZURE_OPENAI_CHAT_DEPLOYMENT before using live mode.")

    if AZURE_OPENAI_API_VERSION == "v1":
        url = f"{AZURE_AI_SERVICES_ENDPOINT.rstrip('/')}/openai/v1/chat/completions"
        body = {"model": AZURE_OPENAI_CHAT_DEPLOYMENT}
    else:
        url = f"{AZURE_AI_SERVICES_ENDPOINT.rstrip('/')}/openai/deployments/{AZURE_OPENAI_CHAT_DEPLOYMENT}/chat/completions?api-version={AZURE_OPENAI_API_VERSION}"
        body = {}

    body.update({
        "messages": [
            {"role": "system", "content": build_system_prompt(instructions, sources)},
            {"role": "user", "content": prompt}
        ],
        "reasoning_effort": reasoning_effort,
        "max_completion_tokens": max_completion_tokens,
        "stream": False
    })
    response = request_json("POST", url, openai_headers(), body)
    message = response.get("choices", [{}])[0].get("message", {})
    return {"answer": message.get("content", ""), "sources": sources, "usage": response.get("usage")}

## 5. Prompt Examples

These examples match the style of the activity guide and the web playground.

In [ ]:
PROMPTS = {
    "warmup": "In one sentence, what can generative AI help a water utility team do?",
    "public_education": "How does drinking water treatment work?",
    "grounded_short": "Use the documents. Answer in one short sentence: what is WTP Minor Improvements? Cite the file.",
    "grounded_table": "Use the documents. Return only a markdown table with exactly 2 rows and 3 columns: category, description, source. Rows: WTP Minor Improvements; Major Mains Bucket. Keep descriptions under 10 words."
}

for name, prompt in PROMPTS.items():
    print(f"{name}: {prompt}")

## 6. Ungrounded Chat

Use this to compare a general model response with a grounded response.

In [ ]:
result = chat(PROMPTS["public_education"], use_grounding=False, max_completion_tokens=600)
pretty_print(result["answer"])

## 7. Grounded Chat With Sources

In [ ]:
result = chat(PROMPTS["grounded_short"], use_grounding=True, reasoning_effort="medium", max_completion_tokens=900)
pretty_print(result["answer"])

print("\nSources")
for index, source in enumerate(result["sources"], start=1):
    location = ""
    if source.get("sourceFile"):
        location += source["sourceFile"]
    if source.get("page"):
        location += f" page {source['page']}"
    print(f"[source {index}] {location}")
    pretty_print(source.get("content", ""), width=96)
    print()

## 8. Structured Output

In [ ]:
result = chat(PROMPTS["grounded_table"], use_grounding=True, reasoning_effort="medium", max_completion_tokens=900)
print(result["answer"])

## 9. Share-Out Prompt

Use this reflection prompt for the interactive learning-sharing component rather than adding more tooling.

In [ ]:
reflection_prompt = "What changed after adding instructions or retrieved documents to the answer?"
print(reflection_prompt)